# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 13
## Model Persistence & Integrasi Big Data Analitik End-to-End

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 13 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Platform e-commerce membangun sistem **deteksi dini transaksi berisiko dibatalkan pelanggan (*cancellation risk*)**. Alur kerja mengintegrasikan seluruh teknologi praktikum semester ini secara *end-to-end*: penyimpanan transaksi mentah di **MongoDB**, ekstraksi & pemodelan klasifikasi (`RandomForestClassifier`) dalam **Spark ML Pipeline**, penyimpanan model terlatih (*model persistence*), pemuatan ulang model pada sesi **SparkSession yang benar-benar baru** tanpa pelatihan ulang, serta ekspor laporan akhir ke format kolumnar **Parquet terpartisi**.

---
## Bagian A. Menyisipkan Data ke MongoDB (Bobot 10%)

Membuat database `ecommerce_p13` dan koleksi `transaksi_mentah`, lalu menyisipkan **1.000 dokumen transaksi** menggunakan modul `random` bawaan Python (agar kompatibel penuh dengan serialisasi BSON PyMongo). Peluang label `dibatalkan` (`0`/`1`) dirancang meningkat seiring bertambahnya `estimasi_pengiriman_hari`.

In [ ]:
import random
import math
from pymongo import MongoClient

# Koneksi ke MongoDB lokal
client = MongoClient("mongodb://localhost:27017/")
db = client["ecommerce_p13"]
koleksi = db["transaksi_mentah"]

# Bersihkan koleksi agar konsisten saat dijalankan ulang
koleksi.delete_many({})

random.seed(1313)
n_dokumen = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]
metode_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

dokumen_transaksi = []
for i in range(n_dokumen):
    estimasi = random.randint(1, 14)
    # Semakin lama estimasi pengiriman, semakin besar peluang transaksi dibatalkan
    prob_batal = 1 / (1 + math.exp(-(estimasi - 7) / 2.2))
    dibatalkan = 1 if random.random() < prob_batal else 0

    dokumen_transaksi.append({
        "order_id": f"ORD13-{1000 + i}",
        "kategori": random.choice(kategori_list),
        "kota": random.choice(kota_list),
        "unit_terjual": random.randint(1, 10),
        "harga_satuan": random.choice([25000, 50000, 75000, 100000, 150000]),
        "metode_pembayaran": random.choice(metode_list),
        "estimasi_pengiriman_hari": estimasi,
        "dibatalkan": dibatalkan,
    })

koleksi.insert_many(dokumen_transaksi)
print(f"Database: {db.name} | Koleksi: {koleksi.name}")
print(f"Total dokumen berhasil disisipkan ke MongoDB: {koleksi.count_documents({})}")
print("Contoh 1 dokumen:", koleksi.find_one({}, {"_id": 0}))

---
## Bagian B. Ekstraksi ke Spark DataFrame (Bobot 10%)

Mengambil seluruh dokumen dari koleksi MongoDB `transaksi_mentah`, mengonversinya menjadi **Spark DataFrame**, lalu menampilkan `printSchema()` dan jumlah barisnya.

In [ ]:
import os
import sys
import pandas as pd

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas13-Sesi1-Training-Yarsya") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Ekstraksi dokumen dari MongoDB (abaikan _id agar langsung bersih)
data_mongo = list(koleksi.find({}, {"_id": 0}))
df_pandas = pd.DataFrame(data_mongo)

# Konversi ke Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

print("=== SKEMA SPARK DATAFRAME ===")
df_spark.printSchema()
print(f"Total baris diekstraksi: {df_spark.count()} baris")
print("\nDistribusi label dibatalkan:")
df_spark.groupBy("dibatalkan").count().orderBy("dibatalkan").show()
df_spark.show(5)

---
## Bagian C. Melatih Model & Menyimpannya (Bobot 25%)

Membangun `Pipeline` klasifikasi: `StringIndexer` (`kategori`, `kota`, `metode_pembayaran`) → `OneHotEncoder` → `VectorAssembler` (+ `unit_terjual`, `harga_satuan`, `estimasi_pengiriman_hari`) → `RandomForestClassifier` (`labelCol="dibatalkan"`).

Melatih model dengan split **80:20** (`seed=99`), mengevaluasi metrik **Accuracy** dan **AUC-ROC**, lalu menyimpan model pipeline ke folder **`model_pembatalan_tersimpan`**.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Tahap Pipeline
idx_kat = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
idx_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")
idx_metode = StringIndexer(inputCol="metode_pembayaran", outputCol="metode_idx")

ohe = OneHotEncoder(
    inputCols=["kategori_idx", "kota_idx", "metode_idx"],
    outputCols=["kategori_vec", "kota_vec", "metode_vec"]
)

assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "estimasi_pengiriman_hari", "kategori_vec", "kota_vec", "metode_vec"],
    outputCol="features"
)

rf = RandomForestClassifier(featuresCol="features", labelCol="dibatalkan", numTrees=30, seed=99)

pipeline_batal = Pipeline(stages=[idx_kat, idx_kota, idx_metode, ohe, assembler, rf])

# Split 80:20 dengan seed=99
train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=99)
print(f"Data training: {train_data.count()} baris | Data testing: {test_data.count()} baris")

# Latih Pipeline
model_batal = pipeline_batal.fit(train_data)
pred_test = model_batal.transform(test_data)

# Evaluasi Accuracy & AUC-ROC
ev_acc = MulticlassClassificationEvaluator(labelCol="dibatalkan", predictionCol="prediction", metricName="accuracy")
ev_auc = BinaryClassificationEvaluator(labelCol="dibatalkan", metricName="areaUnderROC")

acc = ev_acc.evaluate(pred_test)
auc = ev_auc.evaluate(pred_test)
print(f"\n=== HASIL EVALUASI MODEL RANDOM FOREST ===")
print(f"Accuracy : {acc:.4f}")
print(f"AUC-ROC  : {auc:.4f}")

# Simpan model ke disk
path_model = "model_pembatalan_tersimpan"
model_batal.write().overwrite().save(path_model)
print(f"\nModel berhasil disimpan ke folder '{path_model}'.")
!ls -la {path_model}

---
## Bagian D. Simulasi "Minggu Berikutnya" — Sesi Spark Baru (Bobot 20%)

Menutup `SparkSession` lama (`spark.stop()`), membuat **`SparkSession` baru** dari nol, lalu memuat kembali model menggunakan `PipelineModel.load("model_pembatalan_tersimpan")` **tanpa menjalankan `fit()` ulang**. Kemudian memprediksi risiko pembatalan pada **10 transaksi baru**.

In [ ]:
# 1. Tutup SparkSession lama secara eksplisit
spark.stop()
print("SparkSession sesi training (Sesi 1) telah DITUTUP.")

In [ ]:
from pyspark.ml import PipelineModel

# 2. Buat SparkSession BARU (mensimulasikan minggu berikutnya / sesi terpisah)
spark = SparkSession.builder \
    .appName("Tugas13-Sesi2-InferensiMingguDepan-Yarsya") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession BARU aktif:", spark.sparkContext.appName)

# 3. Muat model tersimpan tanpa melatih ulang
model_loaded = PipelineModel.load(path_model)
print(f"Model berhasil dimuat dari '{path_model}'! Jumlah tahapan pipeline: {len(model_loaded.stages)}")

# 4. Buat 10 transaksi baru dengan variasi estimasi pengiriman cepat hingga sangat lama
transaksi_baru = pd.DataFrame([
    {"order_id": "NEW-01", "kategori": "Elektronik",   "kota": "Magelang",   "unit_terjual": 1, "harga_satuan": 150000, "metode_pembayaran": "Transfer Bank", "estimasi_pengiriman_hari": 2},
    {"order_id": "NEW-02", "kategori": "Fashion",      "kota": "Yogyakarta", "unit_terjual": 3, "harga_satuan": 50000,  "metode_pembayaran": "E-Wallet",      "estimasi_pengiriman_hari": 3},
    {"order_id": "NEW-03", "kategori": "Makanan",      "kota": "Semarang",   "unit_terjual": 5, "harga_satuan": 25000,  "metode_pembayaran": "COD",           "estimasi_pengiriman_hari": 1},
    {"order_id": "NEW-04", "kategori": "Rumah Tangga", "kota": "Solo",       "unit_terjual": 2, "harga_satuan": 100000, "metode_pembayaran": "Kartu Kredit",  "estimasi_pengiriman_hari": 5},
    {"order_id": "NEW-05", "kategori": "Kesehatan",    "kota": "Magelang",   "unit_terjual": 4, "harga_satuan": 75000,  "metode_pembayaran": "E-Wallet",      "estimasi_pengiriman_hari": 6},
    {"order_id": "NEW-06", "kategori": "Elektronik",   "kota": "Semarang",   "unit_terjual": 2, "harga_satuan": 150000, "metode_pembayaran": "COD",           "estimasi_pengiriman_hari": 9},
    {"order_id": "NEW-07", "kategori": "Fashion",      "kota": "Solo",       "unit_terjual": 1, "harga_satuan": 100000, "metode_pembayaran": "Transfer Bank", "estimasi_pengiriman_hari": 11},
    {"order_id": "NEW-08", "kategori": "Makanan",      "kota": "Yogyakarta", "unit_terjual": 6, "harga_satuan": 25000,  "metode_pembayaran": "COD",           "estimasi_pengiriman_hari": 12},
    {"order_id": "NEW-09", "kategori": "Rumah Tangga", "kota": "Magelang",   "unit_terjual": 3, "harga_satuan": 75000,  "metode_pembayaran": "Transfer Bank", "estimasi_pengiriman_hari": 13},
    {"order_id": "NEW-10", "kategori": "Kesehatan",    "kota": "Semarang",   "unit_terjual": 2, "harga_satuan": 50000,  "metode_pembayaran": "Kartu Kredit",  "estimasi_pengiriman_hari": 14},
])

df_baru = spark.createDataFrame(transaksi_baru)
hasil_prediksi = model_loaded.transform(df_baru)

print("\n=== HASIL PREDIKSI RISIKO PEMBATALAN 10 TRANSAKSI BARU ===")
hasil_prediksi.select("order_id", "kategori", "kota", "estimasi_pengiriman_hari", "prediction", "probability").show(10, truncate=False)

---
## Bagian E. Menyimpan Laporan Akhir (Bobot 20%)

Menyimpan hasil prediksi bagian D dalam format kolumnar **Parquet** ke folder `laporan_risiko_pembatalan`, dipartisi berdasarkan kolom `kategori`, lalu memverifikasi pembacaannya kembali beserta jumlah barisnya.

In [ ]:
path_laporan = "laporan_risiko_pembatalan"

# Simpan sebagai Parquet terpartisi per kategori
hasil_prediksi.write.mode("overwrite").partitionBy("kategori").parquet(path_laporan)
print(f"Laporan akhir berhasil disimpan ke '{path_laporan}'.")

# Tampilkan struktur direktori partisi
!ls -la {path_laporan}

# Verifikasi membaca kembali dan menghitung jumlah baris
df_verifikasi = spark.read.parquet(path_laporan)
jumlah_baca = df_verifikasi.count()
print(f"\nVerifikasi pembacaan kembali dari Parquet: {jumlah_baca} baris.")
assert jumlah_baca == 10, "Jumlah baris laporan tidak sesuai!"
df_verifikasi.select("order_id", "kategori", "kota", "estimasi_pengiriman_hari", "prediction").orderBy("order_id").show(10)

---
## Bagian F. Kesimpulan & Rekomendasi Operasional (Bobot 15%)

### Alur Kerja End-to-End yang Telah Dibangun
Pipeline analitik deteksi dini pembatalan pesanan ini mengintegrasikan empat komponen inti Big Data dalam satu siklus kerja yang terotomatisasi. Pertama, data transaksi mentah disimpan ke dalam database NoSQL **MongoDB** (`ecommerce_p13.transaksi_mentah`) untuk menampung pencatatan sistem pemesanan secara fleksibel. Kedua, data diekstraksi ke **Apache Spark DataFrame** untuk melewati pemrosesan fitur terstruktur (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`) dan pelatihan model klasifikasi `RandomForestClassifier` dalam satu **ML Pipeline**. Ketiga, model terlatih disimpan menggunakan fitur **Model Persistence** (`model_pembatalan_tersimpan`) sehingga dapat dimuat kembali pada sesi `SparkSession` yang benar-benar terpisah tanpa mengulang proses `fit()`. Keempat, hasil prediksi transaksi baru disimpan secara permanen sebagai berkas **Parquet terpartisi berdasarkan kategori**, yang efisien untuk analitik lanjutan maupun pelaporan.

### Rekomendasi Bisnis Konkret bagi Tim Operasional
Berdasarkan pola yang dipelajari model serta hasil inferensi pada 10 transaksi baru, fitur **`estimasi_pengiriman_hari`** merupakan faktor penentu utama risiko pembatalan: transaksi dengan estimasi di bawah 6 hari cenderung aman (`prediction = 0.0`), sedangkan estimasi pengiriman **>= 8–9 hari ke atas** memiliki probabilitas pembatalan sangat tinggi (`prediction = 1.0`). Oleh karena itu, tim operasional direkomendasikan menetapkan **batas ambang SLA pengiriman maksimal 7 hari** serta mengaktifkan aturan intervensi otomatis (seperti *routing* pesanan ke gudang kota terdekat, kerja sama ekspedisi kilat bersubsidi, atau pengiriman pesan konfirmasi & voucher insentif) setiap kali estimasi sistem melampaui 7 hari guna menekan tingkat pembatalan pelanggan.

In [ ]:
# Tutup koneksi MongoDB dan SparkSession
client.close()
spark.stop()
print("Koneksi MongoDB & SparkSession ditutup dengan aman.")